# CSE 291 / DSC 291 PA3 — Speculative Decoding

In this notebook you will implement and benchmark a single-sequence (batch=1) speculative decoder.

Recap of the algorithm:

1. A small **draft** model proposes `k` tokens autoregressively starting from the current context.
2. The large **target** model verifies the proposal in **one** forward pass (a single batched pass over the `L + k` length sequence).
3. Tokens are accepted greedily up to the first mismatch with the target's argmax. After the first mismatch, the target's own next token is appended and the loop restarts.

Default model pair (public weights, runs on any GPU with >=4 GB VRAM):

- target: `EleutherAI/pythia-1.4b-deduped`
- draft:  `EleutherAI/pythia-160m-deduped`

If you don't have GPU access, the same code paths run on CPU but you won't see a meaningful speedup.

## Setup

In [1]:
import os
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

c:\Users\mesha\anaconda3\envs\cse291pa3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Speculative Decoder

In [2]:
class SpeculativeDecoder:
    def __init__(self, target_model_name: str, draft_model_name: str, device: str = "cuda"):
        """Initialize the speculative decoder with a target and a draft model."""
        self.device = device
        self.target_model, self.target_tokenizer = self.initialize_target_model(target_model_name)
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(draft_model_name)

        assert self.target_tokenizer.get_vocab() == self.draft_tokenizer.get_vocab(), (
            "Target and draft must share a vocabulary"
        )

    def initialize_target_model(self, model_name: str):
        """Load the larger target model with caching enabled."""
        print(f"Loading target model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16
        ).to(self.device)
        model.eval()
        model.config.use_cache = True
        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        """Load the smaller draft model."""
        print(f"Loading draft model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16
        ).to(self.device)
        model.eval()
        model.config.use_cache = True
        return model, tokenizer

    def generate_draft_tokens(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                              num_speculative_tokens: int = 10) -> torch.Tensor:
        with torch.no_grad():
            output = self.draft_model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=num_speculative_tokens,
                do_sample=False,
                pad_token_id=self.draft_tokenizer.pad_token_id,
            )
        return output[:, input_ids.shape[1]:]

    def verify_tokens_vectorized(self, input_ids: torch.Tensor, draft_tokens: torch.Tensor,
                                 attention_mask: torch.Tensor):
        """One target forward pass over prompt + k draft tokens.
        Returns (accepted_draft_tokens, first_mismatch_index, next_token).
        next_token is target's argmax at the mismatch or at the slot past the
        draft if all matched — taken from the same forward pass.
        """
        L = input_ids.shape[1]
        k = draft_tokens.shape[1]

        combined = torch.cat([input_ids, draft_tokens], dim=1)
        combined_mask = torch.cat(
            [attention_mask, torch.ones_like(draft_tokens)], dim=1
        )

        with torch.no_grad():
            logits = self.target_model(combined, attention_mask=combined_mask).logits

        relevant_logits = logits[:, L - 1 : L + k, :]
        target_picks = relevant_logits.argmax(dim=-1)[0]

        matches = target_picks[:k] == draft_tokens[0]
        if matches.all():
            accepted_position = k
            next_token = target_picks[k].item()
            accepted_tokens = draft_tokens[0].tolist()
        else:
            accepted_position = int((~matches).int().argmax().item())
            next_token = target_picks[accepted_position].item()
            accepted_tokens = draft_tokens[0, :accepted_position].tolist()

        return accepted_tokens, accepted_position, next_token

    def speculative_decode(self, prompt: str, max_tokens: int = 100,
                           num_speculative_tokens: int = 8) -> str:
        """Main speculative decoding loop."""
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        prompt_length = input_ids.shape[1]

        total_tokens_generated = prompt_length
        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0
        start_time = time.time()

        eos_token_id = self.target_tokenizer.eos_token_id
        hit_eos = False
        while (total_tokens_generated - prompt_length) < max_tokens and not hit_eos:
            draft_tokens = self.generate_draft_tokens(
                input_ids, attention_mask,
                num_speculative_tokens=num_speculative_tokens,
            )
            total_draft_tokens_proposed += num_speculative_tokens

            accepted, accepted_position, next_token = self.verify_tokens_vectorized(
                input_ids, draft_tokens, attention_mask
            )
            total_draft_tokens_accepted += accepted_position

            new_tokens = accepted + [next_token]
            new_tokens_tensor = torch.tensor(
                [new_tokens], device=self.device, dtype=input_ids.dtype
            )
            input_ids = torch.cat([input_ids, new_tokens_tensor], dim=1)
            attention_mask = torch.cat(
                [attention_mask, torch.ones_like(new_tokens_tensor)], dim=1
            )
            total_tokens_generated += len(new_tokens)

            if eos_token_id is not None and eos_token_id in new_tokens:
                hit_eos = True

        elapsed_time = time.time() - start_time
        acceptance_rate = (
            total_draft_tokens_accepted / total_draft_tokens_proposed
            if total_draft_tokens_proposed > 0 else 0
        )

        print(f"Generated {total_tokens_generated - prompt_length} tokens in {elapsed_time:.2f} seconds")
        print(f"Tokens per second: {(total_tokens_generated - prompt_length) / elapsed_time:.2f}")
        print(f"Draft token acceptance rate: {acceptance_rate:.2%}")

        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    def benchmark(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_runs: int = 3,
        compare_baseline: bool = True,
    ) -> Dict:
        results = {
            "speculative": {"times": [], "tokens_per_second": []},
            "baseline": {"times": [], "tokens_per_second": []} if compare_baseline else None,
        }

        for _ in range(num_runs):
            t0 = time.time()
            output = self.speculative_decode(prompt, max_tokens=max_tokens)
            elapsed = time.time() - t0
            prompt_len = len(self.target_tokenizer(prompt)["input_ids"])
            output_tokens = len(self.target_tokenizer.encode(output)) - prompt_len
            results["speculative"]["times"].append(elapsed)
            results["speculative"]["tokens_per_second"].append(output_tokens / elapsed)

        if compare_baseline:
            for _ in range(num_runs):
                inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)
                t0 = time.time()
                with torch.no_grad():
                    output_ids = self.target_model.generate(
                        input_ids,
                        attention_mask=attention_mask,
                        max_length=input_ids.shape[1] + max_tokens,
                        do_sample=False,
                        pad_token_id=self.target_tokenizer.pad_token_id,
                    )
                elapsed = time.time() - t0
                output_tokens = output_ids.shape[1] - input_ids.shape[1]
                results["baseline"]["times"].append(elapsed)
                results["baseline"]["tokens_per_second"].append(output_tokens / elapsed)

        for method in results:
            if results[method] is not None:
                results[method]["avg_time"] = sum(results[method]["times"]) / num_runs
                results[method]["avg_tokens_per_second"] = (
                    sum(results[method]["tokens_per_second"]) / num_runs
                )
        if compare_baseline:
            results["speedup"] = (
                results["baseline"]["avg_time"] / results["speculative"]["avg_time"]
            )
            results["latency_reduction"] = (
                1 - results["speculative"]["avg_time"] / results["baseline"]["avg_time"]
            ) * 100
        return results


## Test

In [3]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"
draft_model_name = "EleutherAI/pythia-160m-deduped"

decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

test_prompts = [
    "The future of artificial intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}: {prompt}")
    results = decoder.benchmark(prompt=prompt, max_tokens=100, num_runs=3, compare_baseline=True)
    print(f"  Speculative: {results['speculative']['avg_time']:.2f}s, "
          f"{results['speculative']['avg_tokens_per_second']:.2f} tok/s")
    if results['baseline'] is not None:
        print(f"  Baseline:    {results['baseline']['avg_time']:.2f}s, "
              f"{results['baseline']['avg_tokens_per_second']:.2f} tok/s")
        print(f"  Speedup: {results['speedup']:.2f}x  |  Latency reduction: {results['latency_reduction']:.2f}%")

Loading target model: EleutherAI/pythia-1.4b-deduped


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/292 [00:00<04:12,  1.15it/s]

Loading weights:  15%|█▌        | 44/292 [00:00<00:04, 61.46it/s]

...

  Speculative: 2.79s, 38.70 tok/s
  Baseline:    3.77s, 26.53 tok/s
  Speedup: 1.35x  |  Latency reduction: 25.92%

Benchmarking Prompt 3: Write the lyrics to the song 'Happy Birthday'.


Generated 102 tokens in 2.55 seconds
Tokens per second: 40.02
Draft token acceptance rate: 93.75%


Generated 102 tokens in 2.51 seconds
Tokens per second: 40.70
Draft token acceptance rate: 93.75%


Generated 102 tokens in 2.63 seconds
Tokens per second: 38.79
Draft token acceptance rate: 93.75%


  Speculative: 2.56s, 39.82 tok/s
  Baseline:    3.71s, 26.99 tok/s
  Speedup: 1.45x  |  Latency reduction: 30.94%


## Part 3.3 — `num_speculative_tokens` Sweep

Measure acceptance rate and speedup at K ∈ {2, 4, 8, 16}.

In [4]:
# Part 3.3: sweep num_speculative_tokens in {2, 4, 8, 16}
import time

sweep_prompt = "The future of artificial intelligence is"
num_runs = 3
max_tokens = 100

sweep_results = {}

# Baseline (target-only, do_sample=False) measured once — independent of K
base_times = []
for _ in range(num_runs):
    inputs = decoder.target_tokenizer(sweep_prompt, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to(decoder.device)
    attention_mask = inputs["attention_mask"].to(decoder.device)
    t0 = time.time()
    with torch.no_grad():
        decoder.target_model.generate(
            input_ids, attention_mask=attention_mask,
            max_length=input_ids.shape[1] + max_tokens,
            do_sample=False, pad_token_id=decoder.target_tokenizer.pad_token_id,
        )
    base_times.append(time.time() - t0)
avg_base = sum(base_times) / num_runs

print(f"Baseline (target-only, greedy): {avg_base:.2f} s for {max_tokens} tokens")
print()

for K in [2, 4, 8, 16]:
    spec_times, accept_rates = [], []
    for _ in range(num_runs):
        inputs = decoder.target_tokenizer(sweep_prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(decoder.device)
        attention_mask = inputs["attention_mask"].to(decoder.device)
        prompt_len = input_ids.shape[1]

        total = prompt_len
        proposed, accepted_total = 0, 0
        t0 = time.time()
        while total - prompt_len < max_tokens:
            drafts = decoder.generate_draft_tokens(input_ids, attention_mask, K)
            proposed += K
            acc, pos, nxt = decoder.verify_tokens_vectorized(input_ids, drafts, attention_mask)
            accepted_total += pos
            new = acc + [nxt]
            tn = torch.tensor([new], device=decoder.device, dtype=input_ids.dtype)
            input_ids = torch.cat([input_ids, tn], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(tn)], dim=1)
            total += len(new)
            eos = decoder.target_tokenizer.eos_token_id
            if eos is not None and eos in new:
                break
        spec_times.append(time.time() - t0)
        accept_rates.append(accepted_total / proposed if proposed > 0 else 0)

    avg_spec = sum(spec_times) / num_runs
    avg_acc = sum(accept_rates) / num_runs
    sweep_results[K] = {
        "speedup": avg_base / avg_spec,
        "acceptance": avg_acc,
        "spec_time": avg_spec,
    }
    print(f"K={K:>2}: speedup={avg_base/avg_spec:5.2f}x  acceptance={avg_acc:6.1%}  spec={avg_spec:.2f}s")

print()
print("=" * 60)
print(f"{'K':>4}  {'Speedup':>10}  {'Acceptance':>12}")
print("-" * 60)
for K, r in sweep_results.items():
    print(f"{K:>4}  {r['speedup']:>9.2f}x  {r['acceptance']:>11.1%}")


Baseline (target-only, greedy): 3.72 s for 100 tokens



K= 2: speedup= 1.23x  acceptance= 97.1%  spec=3.03s


K= 4: speedup= 1.37x  acceptance= 95.2%  spec=2.72s


K= 8: speedup= 1.46x  acceptance= 91.7%  spec=2.55s


K=16: speedup= 1.42x  acceptance= 85.7%  spec=2.62s

   K     Speedup    Acceptance
------------------------------------------------------------
   2       1.23x        97.1%
   4       1.37x        95.2%
   8       1.46x        91.7%
  16       1.42x        85.7%


## Bonus 3.B — Tree speculation or n-gram lookup decoding (10 pts)

Implement one stronger speculative-decoding variant and benchmark it
against the baseline:

- **Tree / multi-branch speculation** (Medusa / EAGLE-2 style): verify
  several candidate continuations in a single target forward pass.
- **N-gram lookup decoding** (Prompt Lookup Decoding): draft the next
  tokens from an n-gram cache built over the running sequence instead of
  (or in addition to) the draft model.

Re-run the benchmark with your bonus decoder and report the speedup and
acceptance rate in your write-up. See the bonus rubric in `../README.md`.

In [5]:
# Bonus 3.B — Prompt Lookup Decoding (PLD) combined with the draft model.
# When the trailing n-gram of the running sequence has appeared earlier, the
# tokens that followed become the draft proposal (no model call). On a miss,
# fall back to the small draft model.

import time

class PLDSpeculativeDecoder(SpeculativeDecoder):
    def __init__(self, target_model_name, draft_model_name, device="cuda",
                 ngram_size=3):
        super().__init__(target_model_name, draft_model_name, device)
        self.ngram_size = ngram_size
        self.pld_hits = 0
        self.pld_misses = 0

    def generate_draft_tokens(self, input_ids, attention_mask,
                              num_speculative_tokens=10):
        seq = input_ids[0].tolist()
        L = len(seq)
        n = self.ngram_size
        if L >= 2 * n:
            target = seq[-n:]
            for i in range(L - 2 * n, -1, -1):
                if seq[i : i + n] == target:
                    candidate = seq[i + n : i + n + num_speculative_tokens]
                    if len(candidate) == num_speculative_tokens:
                        self.pld_hits += 1
                        return torch.tensor(
                            [candidate], device=self.device,
                            dtype=input_ids.dtype,
                        )
        self.pld_misses += 1
        return super().generate_draft_tokens(
            input_ids, attention_mask, num_speculative_tokens
        )


# Reuse the already-loaded models from the earlier benchmark.
import gc
gc.collect()
torch.cuda.empty_cache()

pld_decoder = PLDSpeculativeDecoder.__new__(PLDSpeculativeDecoder)
pld_decoder.device = decoder.device
pld_decoder.target_model = decoder.target_model
pld_decoder.target_tokenizer = decoder.target_tokenizer
pld_decoder.draft_model = decoder.draft_model
pld_decoder.draft_tokenizer = decoder.draft_tokenizer
pld_decoder.ngram_size = 3
pld_decoder.pld_hits = 0
pld_decoder.pld_misses = 0

bonus_prompts = [
    "The future of artificial intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'.",
]

print("=" * 70)
print("Bonus 3.B — Prompt Lookup Decoding (ngram_size=3)")
print("=" * 70)

for i, prompt in enumerate(bonus_prompts):
    pld_decoder.pld_hits = 0
    pld_decoder.pld_misses = 0
    print(f"\n--- Prompt {i+1}: {prompt!r} ---")
    results = pld_decoder.benchmark(
        prompt=prompt, max_tokens=100, num_runs=3, compare_baseline=True,
    )
    total = pld_decoder.pld_hits + pld_decoder.pld_misses
    pld_rate = pld_decoder.pld_hits / total if total > 0 else 0
    print(f"  Speculative: {results['speculative']['avg_time']:.2f}s, "
          f"{results['speculative']['avg_tokens_per_second']:.2f} tok/s")
    print(f"  Baseline:    {results['baseline']['avg_time']:.2f}s, "
          f"{results['baseline']['avg_tokens_per_second']:.2f} tok/s")
    print(f"  Speedup: {results['speedup']:.2f}x  |  "
          f"Latency reduction: {results['latency_reduction']:.2f}%")
    print(f"  PLD usage:   hits={pld_decoder.pld_hits}, "
          f"misses={pld_decoder.pld_misses}, rate={pld_rate:.1%}")


Bonus 3.B — Prompt Lookup Decoding (ngram_size=3)

--- Prompt 1: 'The future of artificial intelligence is' ---


Generated 100 tokens in 0.82 seconds
Tokens per second: 121.96
Draft token acceptance rate: 91.67%


Generated 100 tokens in 0.87 seconds
Tokens per second: 114.60
Draft token acceptance rate: 91.67%


Generated 100 tokens in 0.92 seconds
Tokens per second: 108.78
Draft token acceptance rate: 91.67%


  Speculative: 0.87s, 115.07 tok/s
  Baseline:    3.72s, 26.88 tok/s
  Speedup: 4.27x  |  Latency reduction: 76.60%
  PLD usage:   hits=30, misses=6, rate=83.3%

--- Prompt 2: 'Write a short story about a robot learning to feel emotions:' ---


...

  Speculative: 1.24s, 87.21 tok/s
  Baseline:    3.75s, 26.67 tok/s
  Speedup: 3.02x  |  Latency reduction: 66.94%
  PLD usage:   hits=27, misses=12, rate=69.2%

--- Prompt 3: "Write the lyrics to the song 'Happy Birthday'." ---


Generated 102 tokens in 0.88 seconds
Tokens per second: 115.36
Draft token acceptance rate: 93.75%


Generated 102 tokens in 0.83 seconds
Tokens per second: 122.90
Draft token acceptance rate: 93.75%


Generated 102 tokens in 0.86 seconds
Tokens per second: 117.98
Draft token acceptance rate: 93.75%


  Speculative: 0.86s, 118.34 tok/s
  Baseline:    3.72s, 26.92 tok/s
  Speedup: 4.31x  |  Latency reduction: 76.78%
  PLD usage:   hits=30, misses=6, rate=83.3%
